# Notebook 3 · The training process: Q-learning by hand

**Time:** about 1½ hours.

**You'll learn**
- how an agent can learn a policy from reward alone
- the balance between exploring and exploiting
- what the discount γ, the learning rate α and the exploration rate ε each do
- why you evaluate on different episodes from the ones you trained on

So far you've written policies by hand. In this notebook the agent works one out for itself, using
about 30 lines of NumPy and no ML library at all.

## The idea

Suppose you knew, for every state $s$ and action $a$, the number

$$Q(s, a) = \text{the total future reward if I take action } a \text{ in state } s \text{ and act well afterwards.}$$

Then the policy would be easy: in each state, pick the action with the biggest $Q$.

**Q-learning** learns these numbers from experience. Each step gives a small piece of evidence: *I was
in state $s$, took action $a$, got reward $r$ and landed in state $s'$.* That suggests a better estimate
for $Q(s, a)$:

$$\text{target} = r + \gamma \max_{a'} Q(s', a')$$

which reads as "the reward I just got, plus the discounted value of the best thing I can do next". We
then nudge the current estimate a little way towards the target:

$$Q(s, a) \leftarrow Q(s, a) + \alpha \,\big(\text{target} - Q(s, a)\big)$$

- **α, the learning rate,** sets the size of the nudge.
- **γ, the discount,** sets how much the future counts. A reward $k$ decisions ahead is weighted by
  $\gamma^k$. With γ = 0.9, a reward 10 decisions (30 minutes) away counts about a third as much as
  one now, and $1/(1-\gamma) = 10$ decisions is a rough "horizon".
- **If the episode terminated,** there is no future, so the target is just $r$. After a truncation the
  future still exists (we just stopped watching), so the γ term stays.

The agent learns from its own guesses, which is called *bootstrapping*. The early estimates are wrong,
but each update pulls them towards something slightly less wrong, because $r$ is real.

## Setup

In [ ]:
import sys
from pathlib import Path

# toy_env.py lives in onboarding/. This makes it importable from both the
# exercise copy (onboarding/) and the solutions copy (onboarding/solutions/).
here = Path.cwd()
sys.path.insert(0, str(here.parent if here.name == "solutions" else here))

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap

from toy_env import (
    ACTION_NAMES,
    ALTITUDE_SCALE,
    EVAL_SEEDS,
    STAY,
    VELOCITY_SCALE,
    AltitudeHoldEnv,
    check,
    evaluate,
    hold_controller,
    plot_flights,
    plot_learning_curve,
    run_episode,
)

env = AltitudeHoldEnv()  # uses the shaped reward from notebook 2
baselines = {
    "do nothing": evaluate(lambda obs: STAY),
    "hold controller": evaluate(hold_controller),
}
pd.DataFrame(baselines).T.round(2)

These are the numbers to beat, scored on the 50 evaluation flights. A flight is 40 decisions, so 40 is
the best possible `total_reward`.

## Part 1 · States

A table needs a finite list of states, but our observations are continuous. So we chop each one into
bins (*discretise* it). We use the altitude error and the velocity, and ignore the ballast left to keep
the table small (more on that at the end).

- **Altitude:** the bins are narrower near the target, where precise control matters most.
- **Velocity:** by the time of each decision, the balloon has settled into one of a few speeds, one for
  each net number of drops (drops minus vents): about 0.66 m/s for one, 0.95 for two, 1.17 for three,
  and the same downwards. The bin edges sit halfway between those speeds, so each one gets a bin of
  its own.

Choosing good bins like this takes knowledge of the problem. That's one of the weaknesses of tables,
which we'll come back to at the end.

In [ ]:
ALT_EDGES_M = np.array([-1000, -600, -350, -250, -100, 0, 100, 250, 350, 600, 1000])
VEL_EDGES_MS = np.array([-1.27, -1.06, -0.8, -0.33, 0.33, 0.8, 1.06, 1.27])
N_ALT_BINS, N_VEL_BINS, N_ACTIONS = len(ALT_EDGES_M) + 1, len(VEL_EDGES_MS) + 1, 3


def discretise(obs):
    """Turn an observation into an (altitude bin, velocity bin) pair of integers."""
    altitude_error = obs[0] * ALTITUDE_SCALE
    velocity = obs[1] * VELOCITY_SCALE
    return int(np.digitize(altitude_error, ALT_EDGES_M)), int(np.digitize(velocity, VEL_EDGES_MS))


Q = np.zeros((N_ALT_BINS, N_VEL_BINS, N_ACTIONS))
print(f"Q-table shape {Q.shape}: {Q.size} numbers to learn")
obs, _ = env.reset(seed=0)
print("first observation", obs, "-> state", discretise(obs))

`Q[state]` gives the three action values for a state, and `Q[state][action]` gives one of them.

## Part 2 · Choosing actions: explore or exploit?

If the agent always picks the action that currently looks best, it never finds out whether the others
are better. The standard fix is **ε-greedy**: with probability ε pick a random action (explore),
otherwise pick the best-looking one (exploit). Training starts at ε = 1, all random, and lowers it to
0.05 as the agent learns.

### ✏️ Exercise 3.1 · ε-greedy

Use `rng.random()` (a random number between 0 and 1) to decide whether to explore,
`rng.integers(N_ACTIONS)` for a random action, and `np.argmax(Q[state])` for the best one. Return a
plain `int`.

In [ ]:
def choose_action(Q, state, epsilon, rng):
    """A random action with probability epsilon, otherwise the action with the highest Q."""
    # YOUR CODE HERE
    raise NotImplementedError


rng = np.random.default_rng(0)
Q_test = np.zeros((N_ALT_BINS, N_VEL_BINS, N_ACTIONS))
Q_test[3, 4] = [0.0, 0.0, 5.0]
greedy_picks = {choose_action(Q_test, (3, 4), 0.0, rng) for _ in range(100)}
random_counts = np.bincount([choose_action(Q_test, (3, 4), 1.0, rng) for _ in range(3000)], minlength=3)
check(greedy_picks == {2}, "with epsilon = 0 it always picks the best action")
check(all(900 < count < 1100 for count in random_counts), "with epsilon = 1 it picks uniformly at random")

## Part 3 · The update

### ✏️ Exercise 3.2 · One Q-learning step

Write `q_update`, which changes `Q[state][action]` in place:

```
target = reward                                     if terminated
         reward + gamma * max(Q[next_state])        otherwise
Q[state][action] += alpha * (target - Q[state][action])
```

In [ ]:
def q_update(Q, state, action, reward, next_state, terminated, alpha, gamma):
    # YOUR CODE HERE
    raise NotImplementedError


Q_test = np.zeros((N_ALT_BINS, N_VEL_BINS, N_ACTIONS))
Q_test[5, 5] = [1.0, 2.0, 0.0]
q_update(Q_test, (4, 4), 2, reward=1.0, next_state=(5, 5), terminated=False, alpha=0.5, gamma=0.9)
check(np.isclose(Q_test[4, 4, 2], 0.5 * (1.0 + 0.9 * 2.0)), "uses reward + gamma * best next value")
q_update(Q_test, (4, 4), 0, reward=1.0, next_state=(5, 5), terminated=True, alpha=0.5, gamma=0.9)
check(np.isclose(Q_test[4, 4, 0], 0.5), "ignores the future after a termination")
check(Q_test[4, 4, 1] == 0.0, "leaves the other actions alone")

## Part 4 · The training loop

### ✏️ Exercise 3.3 · Put it together

The outer loop is written for you. It starts each episode with a fresh training flight and lowers ε
from 1 to 0.05 over the first 60% of training. Fill in the inner loop. Each step:

1. choose an action with `choose_action`
2. step the environment
3. discretise the new observation
4. update `Q` with `q_update`
5. move on to the new state, add the reward to `total`, and stop once the episode is terminated or
   truncated

Training flights use seeds 0–9,999. Evaluation uses `EVAL_SEEDS` (10,000–10,049), so the agent is
always tested on flights it has never seen.

In [ ]:
def train_q_learning(
    env, n_episodes=2000, alpha=0.1, gamma=0.9, epsilon_start=1.0, epsilon_end=0.05, seed=0
):
    """Train a Q-table. Returns the table and each training episode's return."""
    rng = np.random.default_rng(seed)
    Q = np.zeros((N_ALT_BINS, N_VEL_BINS, N_ACTIONS))
    episode_returns = []
    for episode in range(n_episodes):
        progress = episode / (0.6 * n_episodes)
        epsilon = max(epsilon_end, epsilon_start - (epsilon_start - epsilon_end) * progress)
        obs, info = env.reset(seed=int(rng.integers(10_000)))
        state = discretise(obs)
        total, done = 0.0, False
        while not done:
            # YOUR CODE HERE
            raise NotImplementedError
        episode_returns.append(total)
    return Q, np.array(episode_returns)

Train for 2,000 flights. This takes about 20 seconds.

In [ ]:
%%time
Q, returns = train_q_learning(env)

In [ ]:
ax = plot_learning_curve(returns, label="Q-learning")
ax.axhline(baselines["hold controller"]["total_reward"], color="k", ls="--", label="hold controller")
ax.axhline(baselines["do nothing"]["total_reward"], color="grey", ls=":", label="do nothing")
ax.legend()
plt.show()
check(returns[-200:].mean() > 30, "the training return climbs above 30 (out of 40)")

Early on ε is close to 1, so the agent flies at random and does badly, often getting lost. As ε falls
it relies more and more on what it has learnt, and the return climbs. The curve levels off a little
below the hold controller partly because ε never reaches 0: even at the end, 1 in 20 actions is random.

## Part 5 · Evaluation

To score the learned policy we turn exploration off (ε = 0, always take the best action) and fly it on
the evaluation flights.

In [ ]:
def greedy_policy(obs):
    return int(np.argmax(Q[discretise(obs)]))


results = dict(baselines, **{"Q-learning": evaluate(greedy_policy)})
display(pd.DataFrame(results).T.round(2))
check(results["Q-learning"]["in_band"] > 0.75, "on unseen flights it spends over 75% of the time in the band")

flights = [run_episode(env, greedy_policy, seed) for seed in EVAL_SEEDS[4:7]]
plot_flights(flights, labels=["flight 1", "flight 2", "flight 3"], title="Learned Q-table policy")
plt.show()

Two habits here are worth keeping:

- **Evaluate on episodes you didn't train on.** It's the RL version of a held-out test set. The main
  project scores every agent on a frozen set of held-out scenarios for the same reason.
- **Evaluate without exploration.** You want to know how good the learned policy is, not how well it
  does while still experimenting.

## Part 6 · Looking inside the table

The table *is* the policy. Below, each cell is coloured by the action with the highest Q-value. White
cells are states the agent never visited.

In [ ]:
def bin_labels(edges):
    return [f"< {edges[0]:g}"] + [f"{a:g} to {b:g}" for a, b in zip(edges[:-1], edges[1:])] + [f"> {edges[-1]:g}"]


def plot_q_table(Q, title="Greedy action in each state"):
    best = np.argmax(Q, axis=2).astype(float)
    best[np.all(Q == 0, axis=2)] = np.nan  # never visited
    cmap = ListedColormap(["tab:blue", "lightgrey", "tab:orange"])
    cmap.set_bad("white")
    fig, ax = plt.subplots(figsize=(9, 5))
    image = ax.imshow(best.T, origin="lower", cmap=cmap, vmin=-0.5, vmax=2.5, aspect="auto")
    ax.set_xticks(range(N_ALT_BINS), labels=bin_labels(ALT_EDGES_M), rotation=60)
    ax.set_yticks(range(N_VEL_BINS), labels=bin_labels(VEL_EDGES_MS))
    ax.set(xlabel="altitude error (m)", ylabel="vertical velocity (m/s)", title=title)
    colorbar = fig.colorbar(image, ticks=[0, 1, 2])
    colorbar.ax.set_yticklabels(ACTION_NAMES)
    fig.tight_layout()
    plt.show()


plot_q_table(Q)

Read it as a map: left to right is how far below or above the target the balloon is, and bottom to top
is how fast it's sinking or rising. Each row is one of the settled speeds. You should be able to pick
out:

- **At rest** (the middle row): drop when too low, vent when too high, stay near the target.
- **Climbing gently** (the row above): stay while still below the target, because it's already heading
  the right way, then vent to brake once it arrives.
- **Sinking gently** (the row below): roughly the mirror image.
- **Moving fast** (the top and bottom rows): brake, by venting when rising fast and dropping when
  sinking fast.

A few cells look odd, mostly in the fastest rows. Those states come up rarely, so their values are
still rough. Overall it's the same logic as your `velocity_controller`, but nobody told the agent any
physics: it learnt it from reward alone.

## Part 7 · Experiments

### ✏️ Exercise 3.4 · Predict, then run

The cell below trains four agents (about 20 seconds each), changing one setting at a time. **Before you
run it**, write down what you expect each change to do. Then run it and answer the questions.

- `default`: the settings above
- `gamma = 0`: the agent only cares about the very next reward
- `no exploration`: ε = 0 throughout
- `alpha = 0.01`: a learning rate ten times smaller

In [ ]:
experiments = {
    "default": {},
    "gamma = 0": {"gamma": 0.0},
    "no exploration": {"epsilon_start": 0.0, "epsilon_end": 0.0},
    "alpha = 0.01": {"alpha": 0.01},
}
scores = {}
ax = None
for label, settings in experiments.items():
    Q_exp, returns_exp = train_q_learning(AltitudeHoldEnv(), **settings)
    scores[label] = evaluate(lambda obs: int(np.argmax(Q_exp[discretise(obs)])))
    ax = plot_learning_curve(returns_exp, label=label, ax=ax)
ax.legend()
plt.show()
pd.DataFrame(scores).T.round(2)

**Q1.** What does `gamma = 0` do, and why?

*✏️ Write your answer here.*

**Q2.** What goes wrong with `no exploration`?

*✏️ Write your answer here.*

**Q3.** Why is `alpha = 0.01` so much worse after 2,000 episodes?

*✏️ Write your answer here.*

## Part 8 · Where tables run out

- **We chose the bins by hand.** Too coarse and the agent can't tell important states apart. Too fine
  and it needs far more experience to fill the table in.
- **We ignored `ballast_left`.** Adding it with 10 bins would make the table 10 times bigger and 10
  times slower to fill.
- **Stratus's real observation has 144 numbers.** Even with only 2 bins each, that's $2^{144}$ states.

Notebook 4 replaces the table with a neural network, which takes the raw observation, needs no bins,
and can generalise from states it has seen to similar states it hasn't.

🚀 **Stretch**
- Train with notebook 2's sparse reward (1 inside the band, 0 outside) by passing
  `AltitudeHoldEnv(reward_fn=...)`. Does it still learn to hold altitude? How many actions per flight
  does it use compared with the shaped reward, and why? Does that match your answer to notebook 2's Q3?
- Make the bins twice as fine. Is it better or worse after 2,000 episodes? After 6,000?
- Add `ballast_left` to the state as a third table dimension.

## Summary

- Q-learning nudges $Q(s, a)$ towards $r + \gamma \max_{a'} Q(s', a')$, one step of experience at a time.
- ε-greedy balances exploring (finding out) against exploiting (using what you know).
- γ sets how far ahead the agent looks, α how fast it learns, and ε how much it explores.
- Evaluate greedily, on episodes you didn't train on.

**Git checkpoint:** `git add onboarding/03_q_learning.ipynb`, commit and push.